**Generate Predictions**

In [ ]:
import os
import json
import time
from openai import AzureOpenAI
from tqdm.auto import tqdm

TEST_INPUT_PATH = "test.jsonl"
PRED_OUTPUT_PATH = "test_predictions_sft1.jsonl"

client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version="2025-03-01-preview",
)


SYSTEM_PROMPT = """You are a Haskell coding assistant.
Return only Haskell code.
Do not include markdown fences.
Do not include explanations."""

def generate_completion(prompt: str) -> str:
    response = client.chat.completions.create(
        model='1-nano-2025-04-14-supervised-fine-tuning',
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

num_errors = 0

num_lines = sum(1 for _ in open(TEST_INPUT_PATH, "r", encoding="utf-8"))

with open(TEST_INPUT_PATH, "r", encoding="utf-8") as f_in, \
     open(PRED_OUTPUT_PATH, "w", encoding="utf-8") as f_out:

    for i, line in enumerate(tqdm(f_in, total=num_lines, desc="Generating test predictions")):
        ex = json.loads(line)

        try:
            pred = generate_completion(ex["prompt"])
        except Exception as e:
            pred = ""
            num_errors += 1
            print(f"\nError on example: {e}")
            time.sleep(2)

        out = {
            "prompt": ex["prompt"],
            "reference_solution": ex["reference_solution"],
            "prediction": pred,
        }

        for k, v in ex.items():
            if k not in out:
                out[k] = v

        f_out.write(json.dumps(out, ensure_ascii=False) + "\n")

print(f"Saved predictions to {PRED_OUTPUT_PATH}")
print(f"Errors: {num_errors}")

**Run Predictions**

In [ ]:
!apt-get update
!apt-get install -y ghc

import json
import subprocess
import tempfile
import os
from tqdm.auto import tqdm

TEST_RAW_PATH = "test_raw.jsonl"
PRED_PATH = "test_predictions.jsonl"
RESULTS_PATH = "sft_evaluation.json"

MAX_EXAMPLES = 200
TIMEOUT_SECONDS = 20
PRINT_EVERY = 20

COMMON_SOLUTION_IMPORTS = """module Solution where

import Data.List
import Data.Ord
import Data.Maybe
import Data.Char
import Data.Function
import qualified Data.Map.Strict as Map
import qualified Data.Map as MapLazy
import qualified Data.Set as Set
import qualified Data.List as List
import qualified Data.Ord as Ord
import System.IO.Unsafe
import Data.IORef
"""

def run_tests(solution_code, tests):
    with tempfile.TemporaryDirectory() as tmpdir:
        solution_path = os.path.join(tmpdir, "Solution.hs")
        main_path = os.path.join(tmpdir, "Main.hs")

        solution_module = COMMON_SOLUTION_IMPORTS + "\n" + solution_code

        with open(solution_path, "w", encoding="utf-8") as f:
            f.write(solution_module)

        test_defs = []
        test_checks = []

        for i, test in enumerate(tests):
            test = test.strip()
            test_defs.append(f"test_{i} :: Bool")
            test_defs.append(f"test_{i} = ({test})")
            test_defs.append("")
            test_checks.append(
                f'  putStrLn $ (if test_{i} then "PASS_{i}" else "FAIL_{i}")'
            )

        if not test_checks:
            test_checks = ['  putStrLn "NO_TESTS"']

        main_code = "\n".join([
            "module Main where",
            "import Solution",
            "import qualified Data.Map.Strict as Map",
            "import qualified Data.Map as MapLazy",
            "import qualified Data.Set as Set",
            "import qualified Data.List as List",
            "import qualified Data.Ord as Ord",
            "",
            *test_defs,
            "main :: IO ()",
            "main = do",
            *test_checks
        ])

        with open(main_path, "w", encoding="utf-8") as f:
            f.write(main_code)

        try:
            result = subprocess.run(
                ["runghc", "-i" + tmpdir, main_path],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stdout": "",
                "stderr": f"Execution timed out after {TIMEOUT_SECONDS} seconds.",
                "main_code": main_code,
            }

        if result.returncode != 0:
            return {
                "compiled": False,
                "passed": 0,
                "total": len(tests),
                "stdout": result.stdout,
                "stderr": result.stderr,
                "main_code": main_code,
            }

        passed = sum(
            1 for line in result.stdout.splitlines()
            if line.startswith("PASS_")
        )

        return {
            "compiled": True,
            "passed": passed,
            "total": len(tests),
            "stdout": result.stdout,
            "stderr": result.stderr,
            "main_code": main_code,
        }

with open(TEST_RAW_PATH, "r", encoding="utf-8") as f:
    test_examples = [json.loads(line) for line in f]

with open(PRED_PATH, "r", encoding="utf-8") as f:
    pred_examples = [json.loads(line) for line in f]

if len(test_examples) != len(pred_examples):
    print(
        f"Warning: test file has {len(test_examples)} examples but prediction file has {len(pred_examples)} examples."
    )

num_pairs = min(len(test_examples), len(pred_examples), MAX_EXAMPLES)
results = []

for i in tqdm(range(num_pairs), desc="Evaluating predictions", dynamic_ncols=True):
    test_ex = test_examples[i]
    pred_ex = pred_examples[i]

    prediction = pred_ex.get("prediction", "")
    tests = test_ex.get("translated_test_cases", [])

    eval_result = run_tests(prediction, tests)

    results.append({
        "idx": i,
        "compiled": eval_result["compiled"],
        "tests_passed": eval_result["passed"],
        "tests_total": eval_result["total"],
        "had_no_tests": len(tests) == 0,
        "stderr": eval_result["stderr"][:2000],
    })

    if (i + 1) % PRINT_EVERY == 0 or (i + 1) == num_pairs:
        compiled_so_far = sum(r["compiled"] for r in results)
        passed_so_far = sum(r["tests_passed"] for r in results)
        total_so_far = sum(r["tests_total"] for r in results)

        compile_rate_so_far = compiled_so_far / len(results) if results else 0.0
        test_rate_so_far = passed_so_far / total_so_far if total_so_far else 0.0

        print(
            f"\n[{i + 1}/{num_pairs}] "
            f"compile rate so far: {compiled_so_far}/{len(results)} = {compile_rate_so_far:.3f} | "
            f"test pass rate so far: {passed_so_far}/{total_so_far} = {test_rate_so_far:.3f}"
        )

num_examples = len(results)
compile_successes = sum(r["compiled"] for r in results)
total_passed = sum(r["tests_passed"] for r in results)
total_tests = sum(r["tests_total"] for r in results)

print("===== OVERALL EVALUATION =====")
if num_examples:
    print(f"Examples evaluated: {num_examples}")
    print(f"Compile success: {compile_successes}/{num_examples} = {compile_successes/num_examples:.3f}")
else:
    print("Examples evaluated: 0")

if total_tests:
    print(f"Functional test pass rate: {total_passed}/{total_tests} = {total_passed/total_tests:.3f}")
else:
    print("Functional test pass rate: N/A")

print("\n===== PER-EXAMPLE SUMMARY =====")
for r in results:
    print(f"\n--- Example {r['idx']} ---")
    print("Compiled:", r["compiled"])
    print(f"Tests passed: {r['tests_passed']}/{r['tests_total']}")
    if not r["compiled"]:
        print("Error:")
        print(r["stderr"])

summary = {
    "examples_evaluated": num_examples,
    "compile_successes": compile_successes,
    "compile_rate": (compile_successes / num_examples) if num_examples else None,
    "tests_passed": total_passed,
    "tests_total": total_tests,
    "functional_pass_rate": (total_passed / total_tests) if total_tests else None,
}

output = {
    "summary": summary,
    "per_example_results": results
}

with open(RESULTS_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2)

print(f"\nSaved evaluation results to {RESULTS_PATH}")